# 第3回 欠測値への対応や標準化など前処理の実行（解答例・教員用）


---


## 前回からの接続


第1・2回は、自転車シェアの運営者として、明日の利用台数を当ててきました。第2回では、データを訓練とテストに分けて、まだ見ていないデータで実力を測るという手順を整えました。ただ、自転車のデータは、欠けた値もなく、列の型もそろった、整えられたデータでした。

今日から題材が変わります。現場のデータには、極端な値が混ざり、値が欠け、文字の列が並び、列ごとに目盛りがばらばらです。そのままでは式に入れられないデータを、式に入れられる形に整えます。この作業を前処理（preprocessing）と呼び、今日はこれを学びます。


---


## 今日の分析目標


あなたは民泊の新しいホストです。自分のマンションの一室を貸切で出すとき、1泊いくらの値段を付ければいいかに答える。

東京のAirbnbに掲載された約3万2千件の部屋のデータから、似た部屋の料金を手がかりに値付けします。極端な値・欠けた値・文字の列・ばらばらの目盛りを一つずつ確かめて整え、整えるたびに、まだ見ていない部屋での外れ幅がどう変わるかを確かめます。もう1つのねらいは、同じ処理を関数（`def`）にまとめ、訓練とテストに同じように当てられるようになることです。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは、民泊を始めたばかりの新しいホストです。東京の自分のマンションの一室を、貸切で出したいと考えています。定員や寝室の数は、もう決まっています。

決めたいのは、掲載するときの1泊の値段です。その値段が、似た部屋の相場からどれくらい外れうるかも、知っておきたいところです。


### 手元のデータ

Inside Airbnbが公開している、東京のAirbnbの掲載32,361件です（2026年6月30日時点）。1行が1つの部屋で、地区・部屋のタイプ・定員・寝室数などが並びます。答えの列`price`は、その部屋の1泊の料金（円）です。


### 最後に出す答え

あなたの部屋に付ける1泊の値段（円）と、その値段が、まだ見ていない部屋で平均何円外れるかです。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 極端な値や、目標と別の種類の行を外すかどうかを、分布を見て判断し、その理由を言葉で説明できる
- 何度も使う計算を`def`で関数にまとめ、引数と戻り値を読み書きできる
- 欠測（missing value）の量と欠けた理由を確かめ、行を落とすか補完（imputation）するかを選べる
- 文字の列に番号を振って式に入れることの危うさを、実際の結果で説明できる
- 距離を使う手法では標準化（standardization）で当たりがよくなることを、実際の外れ幅で確かめられる
- 補う値や標準化の物差しを、訓練データだけで決める理由（テストの情報を学習に漏らさない順番）を説明できる


---


## 0. 設定


### 最初に実行する：道具を呼び出す設定のセル

- 役割：このノートで使う道具を呼び出す
- 入力：なし
- 出力：「セットアップ完了」の表示

今日は、線形回帰（linear regression）に加えて、似た部屋を探すKNN（`KNeighborsRegressor`）と、欠けた値を補う`SimpleImputer`、目盛りをそろえる`StandardScaler`を使います。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
try:
    import japanize_matplotlib
except Exception:   # japanize が動かなくなったときの保険：同梱フォントを直接登録する
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None) if spec else None
    if ttf:
        fm.fontManager.addfont(str(ttf))
        plt.rcParams['font.family'] = fm.FontProperties(fname=str(ttf)).get_name()

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
print('セットアップ完了')


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は ① と ② の前処理が中心です。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = i in (1, 2)
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


---


## 1. どの部屋を手がかりにするか


値段は、似た部屋の料金を手がかりに付けます。その前に、データにどんな部屋が並んでいるかを確かめます。東京のAirbnbの掲載データ（Inside Airbnb、2026年6月30日時点）を読み込みます。1行が1つの部屋で、`price`が1泊の料金（円）です。


### お手本：読み込んで、料金の形を見る

- 役割：データを読み込み、料金の要約とヒストグラムを出す
- 入力：データの置き場所とファイル名
- 出力：表`df`と、料金の要約・ヒストグラム

`plt.hist`の`range=(0, 300000)`は、図に描く範囲を0〜30万円に決める引数です。料金は545万円まであるので、全部を描くと山が左端につぶれてしまいます。


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
df = pd.read_csv(f'{DATA_DIR}/airbnb_tokyo.csv')
print(len(df), '件')
print(df['price'].describe().round(0))
plt.hist(df['price'], bins=100, range=(0, 300000), color='#0066cc')
plt.xlabel('1泊の料金（円）'); plt.ylabel('部屋の数')
plt.show()


列は、地区（`neighbourhood`）・部屋のタイプ（`room_type`）のような文字の列と、定員（`accommodates`）・寝室数（`bedrooms`）・ベッド数（`beds`）・バスルーム数（`bathrooms`）・最低宿泊数（`minimum_nights`）・年間の予約可能日数（`availability_365`）・レビュー数（`number_of_reviews`）・ホストの掲載数（`host_listings_count`）のような数の列が混ざっています。

料金の中央値は約2万円なのに、最大は545万円。ヒストグラムは2万円あたりに山が1つあり、右に長い尾が伸びています（図の右端の30万円より先にも、545万円まで続きます）。1泊数十万〜数百万円の部屋としては、一棟貸しの大きな家や、特定の日だけ値段を上げた掲載などが考えられます。こうした部屋は、ふつうの部屋とは別の市場にあると考えたほうがよさそうです。「新しいホストが、ふつうの部屋に値段を付ける」という今日の目標の手がかりになりません。大事なのは、外す理由を目標から説明できることと、どこで線を引いたかを書き残すことです。今日は「1泊10万円以下」に絞ります。


### おさらい：条件で絞る

`df[ ]`にTrue/Falseの並びを入れると、Trueの行だけが残ります。

```python
df[df['price'] <= 100000]      # 10万円以下の部屋だけ
```


### 変えてみる：10万円以下の、貸切の部屋に絞る


In [ ]:
df = df[df['price'] <= 100000]                  # 10万円以下の部屋だけ
print(df.groupby('room_type')['price'].agg(['median', 'count']))
df = df[df['room_type'] == 'Entire home/apt']   # 貸切の部屋だけ
df = df.reset_index(drop=True)                  # 行の番号を振り直す
print(len(df), '件')


部屋のタイプごとに見ると、貸切（Entire home/apt）・個室（Private room）・相部屋（Shared room）で料金の水準がまるで違います。今日のあなたは貸切で出すホストなので、手がかりにする部屋を貸切だけに絞りました。これも「目標に照らして、使う行を決める」前処理です。26,433件になりました。

ここで、第2回と同じく、先に訓練とテストに分けておきます。このあとの前処理は、すべて訓練データだけを見て決めます。


In [ ]:
tr, te = train_test_split(df, test_size=0.2, random_state=42)
y_tr = tr['price'].values
y_te = te['price'].values
print(len(tr), len(te))


---


## 2. 物差しを、関数にまとめる


1節で、手がかりにする部屋を貸切の26,433件に絞り、訓練とテストに分けました。では、整えた結果をどう測ればよいでしょうか。今日は、整えるたびに「まだ見ていない部屋で、何円外れたか」を何度も測ります。同じ計算を毎回書くのは大変です。そこで、計算を関数にまとめます。


### お手本：外れ幅の平均を関数にする

- 役割：実際の料金と予測から、外れ幅の平均（円）を出す
- 入力：実際の値`y`と、予測`pred`
- 出力：外れ幅の平均（整数に丸めた円）

`def 名前(引数):`で関数を作ります。引数は関数が受け取るもの（入力）、`return`の後ろが関数が返すもの（戻り値、出力）です。これまで「役割・入力・出力」と読んできたかたまりを、そのまま1つの名前にしたものが関数です。関数の中身は、字下げした行です。


In [ ]:
def mae(y, pred):          # 外れ幅の平均（第1回と同じ物差し）
    return round(np.abs(y - pred).mean())


このセルを実行しても、何も表示されません。関数を作っただけだからです。使うには、名前にかっこを付けて呼び出します。


### 変えてみる：関数を呼び出す

まず基準として、毎回「訓練の料金の中央値」と答えたら、テストで何円外れるかを測ります。


In [ ]:
base = np.median(y_tr)          # 答え：どの部屋も、訓練の中央値
print(base)
print(mae(y_te, base))          # 関数を呼び出す


何も工夫せずに答えると、平均で10,756円外れます。これを基準に、整えるたびにどれだけ縮むかを見ていきます。


---


## 3. 欠けた値を、落とすか補うか


何も工夫しない答えの外れ幅、10,756円が基準になりました。ここからデータを整えていきます。まずは欠けた値です。数の列には、ところどころ空欄（欠測）があります。どの列が、どれだけ欠けているでしょうか。


### TODO①：列ごとの欠測の数と割合を表示する

`df.isna()`は各マスが欠けているかをTrue/Falseで表します。`.sum()`で列ごとにTrueの数を、`.mean()`でTrueの割合を出せます。欠測の数と割合（%）を表示してください。


In [ ]:
# TODO: 列ごとの欠測の数と、割合（%）を表示する

# 解答例①：列ごとの欠測の数と割合
print(pd.DataFrame({'欠測の数': df.isna().sum(),
                    '割合(%)': (df.isna().mean() * 100).round(1)}))


バスルーム数（`bathrooms`）が6.3%、寝室数（`bedrooms`）が4.9%、ベッド数（`beds`）が3.9%欠けています。レビューの点数（`review_scores_rating`）は11%です。

欠測を扱う前に、なぜ欠けたのかを考えます。レビューの点数が欠けている部屋は、ぴったりレビューが0件の部屋でした。まだ誰も泊まっていないので、点数が付きようがないのです。このような「理由のある欠け方」を平均などで埋めてしまうと、「まだ誰も泊まっていない」という大事な情報が消えます。今日は点数の列を使わずに進めます。


### お手本：欠けたバスルーム数の部屋を見る

- 役割：バスルーム数が欠けた訓練の部屋を取り出し、文字の列`bathrooms_text`に何と書いてあるかを数える
- 入力：訓練の表`tr`
- 出力：欠けた件数と、`bathrooms_text`の中身の多い順

このデータには、バスルームを文字で書いた列`bathrooms_text`（例: `'1 bath'`・`'1.5 baths'`・`'Half-bath'`）もあります。`value_counts()`は、中身ごとの件数を多い順に並べるメソッドです。


In [ ]:
miss = tr['bathrooms'].isna()      # バスルーム数が欠けた部屋は True
print(miss.sum(), '件')
print(tr[miss]['bathrooms_text'].value_counts().head())


欠けた1,276件は、どれも`bathrooms_text`に数が書いてあります（`'0 baths'`の部屋は、数の列ではすべて欠けになっていました）。別の列から本当の値を戻せる欠けです。それなら、中央値で推測して埋めるより、文字から戻すほうが確かです。


### 変えてみる：文字からバスルーム数を戻す

- 役割：`bathrooms_text`の文字から数を読み取り、欠けたバスルーム数をその数で埋める
- 入力：表（`tr`・`te`・`df`）
- 出力：バスルーム数の欠けが埋まった表

`bath_num`は、文字1つを数1つに直す関数です。`'1.5 baths'.split(' ')`は空白で切り分けて`['1.5', 'baths']`にし、`[0]`で最初の`'1.5'`を取り、`float( )`で小数の数にします。`'Half-bath'`（シャワーのない洗面所など、半分のバスルーム）だけは数が書いていないので、`if`で0.5を返します。

`map(関数)`は、列の中身を1つずつ関数に通して、変換した列を返すメソッドです。`fillna`に列を渡すと、欠けたマスだけが、同じ行のその値で埋まります。`assign(列名=値)`は、その列を置き換えた新しい表を返します。

1部屋ずつ、その部屋の文字を読むだけで、ほかの部屋の値は使わないので、訓練・テスト・全体のどれに当ててもかまいません（全体の`df`は6節で使うので、同じように直しておきます）。


In [ ]:
def bath_num(text):            # '1.5 baths' → 1.5
    if text == 'Half-bath':
        return 0.5
    return float(text.split(' ')[0])

def fill_bath(d):              # 欠けたバスルーム数を、文字から戻す
    from_text = d['bathrooms_text'].map(bath_num)
    return d['bathrooms'].fillna(from_text)

tr = tr.assign(bathrooms=fill_bath(tr))
te = te.assign(bathrooms=fill_bath(te))
df = df.assign(bathrooms=fill_bath(df))
print(tr['bathrooms'].isna().sum(), te['bathrooms'].isna().sum())


バスルーム数の欠けは、訓練もテストも0件になりました。


### 変えてみる：寝室数の欠けた部屋は、どんな部屋か

寝室数（`bedrooms`）が欠けた部屋も、ほかの部屋と比べてみます。お手本の`miss`と同じように、欠けているかのTrue/Falseを作り、それで`groupby`します。


In [ ]:
miss_bed = tr['bedrooms'].isna()   # 寝室数が欠けた部屋は True
print(tr.groupby(miss_bed)[['accommodates', 'price']].median())
print('寝室数が0の部屋:', (tr['bedrooms'] == 0).sum(), '件')


寝室数が欠けた部屋（`True`の行）は、定員の中央値が2人、料金の中央値が15,960円で、欠けていない部屋（4人、20,464円）より小さく安い部屋に偏っています。しかも、寝室数が0の部屋は1件もありません。寝室のないワンルームが、寝室数「0」ではなく欠けとして記録されているのかもしれません。これも理由のある欠け方の疑いがあります。


### 欠けていた印を残す

それでも寝室数は、料金の手がかりとして使いたい列です。そこで、欠けた値はこのあと中央値で補いますが、その前に、欠けていたことの印を別の列`bedrooms_missing`（欠けていたら1、いなければ0）として残します。こうすると、補った値（推測）の部屋と、本当の値の部屋を、式が区別できます。`astype(int)`は、True/Falseを1/0の数に直すメソッドです。


In [ ]:
tr = tr.assign(bedrooms_missing=tr['bedrooms'].isna().astype(int))
te = te.assign(bedrooms_missing=te['bedrooms'].isna().astype(int))
df = df.assign(bedrooms_missing=df['bedrooms'].isna().astype(int))
print(tr['bedrooms_missing'].sum(), '件に印')


### 落とすか、補うか

寝室数やベッド数の欠けには、二つの手があります。

- 行を落とす：どこかが欠けた部屋を丸ごと使わない。手軽だが、データが減る
- 補う（補完）：欠けた値を、訓練データの中央値などで埋める。データは減らない

中央値を使うのは、極端な値に引きずられにくいからです（寝室数はほとんどが1か2ですが、10の部屋も混ざっています）。ただし、補った値はあくまで推測です。寝室数が欠けた部屋は小さい部屋に偏っていたので、中央値（寝室1）は、その部屋の本当の値とずれているかもしれません。だから、印の列を残しておきます。


### お手本：訓練の中央値で補う

- 役割：数の列の欠けた値を、訓練データの中央値で埋める
- 入力：訓練とテストの表`tr`・`te`
- 出力：欠けのない配列`X_tr`・`X_te`

`fillna(値)`は、欠けたマスをその値で置き換えるメソッドです。列ごとの中央値を渡すと、列ごとにその値で埋めます。数の列には、印の列`bedrooms_missing`も入れます。


In [ ]:
num = ['accommodates', 'bedrooms', 'beds', 'bathrooms',
       'minimum_nights', 'availability_365', 'number_of_reviews',
       'host_listings_count', 'bedrooms_missing']
med = tr[num].median()                 # 補う値は、訓練だけで決める
X_tr = tr[num].fillna(med).values
X_te = te[num].fillna(med).values      # テストも、訓練の中央値で
print(med)


### 変えてみる：落とした場合と、補った場合を比べる

同じ線形回帰の式を、欠けた行を落として作った場合と、中央値で補って作った場合で比べます。2節の`mae`を2回呼ぶだけです。違うのは訓練の扱いだけで、テストはどちらも、お手本で訓練の中央値を使って補った`X_te`で測ります。

`notna()`は欠けていないマスをTrueで表し、`.all(axis=1)`は1行ずつ、その行の列が全部Trueかを調べます。`axis=1`は「行ごとに、列をまたいでまとめる」という意味です（5節で使う`axis=0`は「列ごとに、行をまたいでまとめる」）。最後の行の`coef_[-1]`は、係数の並びの最後（ここでは印の列）の係数です。`[-1]`は、後ろから1つ目を取り出します。


In [ ]:
ok = tr[num].notna().all(axis=1).values      # 欠けのない行は True
m_drop = LinearRegression().fit(tr[num].values[ok], y_tr[ok])
m_num = LinearRegression().fit(X_tr, y_tr)
print('落とした:', mae(y_te, m_drop.predict(X_te)), '円')
print('補った  :', mae(y_te, m_num.predict(X_te)), '円')
print('使えた行:', ok.sum(), '/', len(ok))
print('印の係数:', round(m_num.coef_[-1]), '円')


行を落とすと、訓練21,146件のうち使えるのは19,321件だけで、8.6%（1,825件）を捨てることになります。外れ幅は8,822円。中央値で補うと8,807円で、少しだけ良くなりました。どちらも基準の10,756円から2千円近く縮んでいます。

差は小さくても、補うやり方には利点があります。新しいホストが寝室数を書き忘れていたら、どちらのやり方でも、値付けするには補うしかありません（テストもそうして測りました）。それなら訓練でも同じように補えば、データを捨てずに済みます。

印の列の係数は−140円でした。定員などがそろっていれば、寝室数が欠けていた部屋だけが特に安い、というわけではないようです。欠けた部屋が安く見えたのは、主に定員が小さいからでした。このデータでは、中央値で補っても、式が大きくゆがむことはなさそうです。印を残したおかげで、そのことを確かめられました。

scikit-learnの`SimpleImputer(strategy='median')`も、同じことをする道具です。訓練で`fit`すると、`statistics_`に中央値が入ります。


In [ ]:
imp = SimpleImputer(strategy='median').fit(tr[num])
print(imp.statistics_)                          # 上の med と同じか？


---


## 4. 文字の列に、番号を振ってよいか


3節で数の列の欠けを補い、式の外れ幅は8,807円まで縮みました。まだ使っていない手がかりに、文字の列の地区があります。同じ貸切でも、地区（`neighbourhood`）によって料金は違いそうです。


### TODO②：地区ごとの料金の中央値を見る

訓練データ`tr`を地区ごとにまとめ、料金の中央値と部屋の数を求めてください。そして、部屋が30件以上ある地区について、中央値の安い順に並べて表示してください（地区の扱いも、訓練だけを見て決めます）。


In [ ]:
# TODO: 訓練データで、地区ごとの料金の中央値と部屋の数を求め、

# 解答例②：地区ごとの料金の中央値（訓練データで）
by_area = tr.groupby('neighbourhood')['price'].agg(
    ['median', 'count'])
print(by_area[by_area['count'] >= 30].sort_values('median'))


安い大田区（Ota Ku）は中央値で約1万5千円、高い千代田区（Chiyoda Ku）は約2万9千円と、同じ貸切でも2倍近く違います。地区は料金に関係していそうです。

でも、地区は文字の列なので、そのままでは式に入れられません。手っ取り早いのは、49の地区に1〜49の番号を振ることです。


### おさらい：for文

`for 名前 in 並び:`は、並びの中身を1つずつ名前に入れながら、下の字下げした行を繰り返します。

```python
for i in range(3):
    print(i)          # 0, 1, 2 と表示される
```


### お手本：地区に番号を振る関数

- 役割：地区の並べ方`order`を受け取り、地区の名前から番号を引ける辞書を返す
- 入力：地区の名前の並び`order`
- 出力：辞書`{地区の名前: 番号}`

辞書（`{}`）は、名前と値の組を覚えておく入れ物です。`code['Ota Ku']`のように、名前で値を引けます。`enumerate(order)`は、並びの中身に0, 1, 2, … の番号を付けて取り出します。番号の振り方も、訓練データ`tr`の地区の名前から決めます。


In [ ]:
def area_code(order):     # 地区の並べ方 order に沿って番号を振る
    code = {}
    for i, name in enumerate(order):
        code[name] = i + 1          # 1番から
    return code

order = sorted(tr['neighbourhood'].unique())   # アルファベット順
c = area_code(order)
print(c['Adachi Ku'], c['Ota Ku'])


### 変えてみる：地区の名前を番号に置き換える

3節では、`map`に関数を渡して、列の中身を1つずつ変換しました。`map`に辞書を渡すと、列の中身を辞書で引いた値に置き換えます。


In [ ]:
print(tr['neighbourhood'].map(c).head(3))   # 名前 → 番号


### 変えてみる：振り方を変えて、結果を比べる

同じ関数に、別の並べ方（部屋の多い順）を渡します。関数にしておくと、引数を変えるだけで同じ処理をやり直せます。

- `orders`は、値に並びを入れた辞書です。`orders.items()`で、名前（`label`）と並び（`order`）の組を1つずつ取り出せます
- `np.c_[X_tr, 列]`は、配列の右に列を横に足す書き方です。数の列の右に、地区の番号の列を足します
- `m.coef_[-1]`は、最後の列、つまり地区の番号の列の係数です


In [ ]:
orders = {'アルファベット順': sorted(tr['neighbourhood'].unique()),
          '部屋の多い順': tr['neighbourhood'].value_counts().index}
for label, order in orders.items():
    c = area_code(order)
    Xc_tr = np.c_[X_tr, tr['neighbourhood'].map(c)]   # 番号の列を足す
    Xc_te = np.c_[X_te, te['neighbourhood'].map(c)]
    m = LinearRegression().fit(Xc_tr, y_tr)
    pred = m.predict(Xc_te)
    print(label, '→', mae(y_te, pred), '円',
          ' 番号1つあたり', round(m.coef_[-1]), '円')


アルファベット順に番号を振ると外れ幅は8,708円、部屋の多い順では8,748円です。外れ幅の差は40円ほどですが、番号の係数を見ると、アルファベット順では番号が1増えると+114円、部屋の多い順では−216円と、向きまで逆になっています。同じ地区の情報なのに、番号の振り方を変えただけで、式の読み方が変わってしまうのです。

式は「番号が1増えると料金がいくら変わる」と、番号の大小に意味があるかのように読みます。でも、アルファベット順で足立区が1番になる、といった番号には、そんな意味はありません（第1回の「区分の番号」と同じ問題です）。番号なし（3節の8,807円）より外れ幅が少し縮んでも、こう読めてしまう式では、値付けの理由を説明できません。

文字の区分を式に入れる方法もありますが、注意点が多くあります。今日は地区を式に入れず、数の列だけで進めます。


---


## 5. 目盛りをそろえる（標準化）


地区は、今日は入れないことにしました。ここまでは1本の式で値付けし、外れ幅は8,807円でした。ここで、式とは別の値付けのやり方を試します。似た条件の部屋の料金の平均で答えるやり方です。第1回の「似た日の平均」と同じ発想で、k近傍法（KNN）と呼びます。「似ている」は、数の列の値どうしの距離で測ります。


### お手本：似た部屋10件の平均で答える

- 役割：数の列の距離がいちばん近い訓練の部屋10件を探し、その料金の平均を答えにする
- 入力：`X_tr`・`y_tr`・`X_te`
- 出力：テストでの外れ幅


In [ ]:
knn_raw = KNeighborsRegressor(n_neighbors=10).fit(X_tr, y_tr)
print(mae(y_te, knn_raw.predict(X_te)), '円')
print(pd.DataFrame(X_tr, columns=num).std().round(1))   # ばらつき


外れ幅は9,526円で、3節の式（8,807円）より悪くなりました（環境によって1〜10円ほどずれることがあります。同じ距離の部屋が何件も並んだとき、どれを選ぶかが環境で少し違うためです）。なぜ式より悪いのでしょうか。列ごとのばらつき（標準偏差）を見ると、年間の予約可能日数は92、ホストの掲載数は75、レビュー数は67もあるのに、寝室数は0.9、バスルーム数は0.4しかありません。原因は目盛りです。距離をそのまま測ると、目盛りの大きい列のちがいだけで、「似ている」かどうかがほぼ決まってしまいます。

標準化は、各列から平均を引き、標準偏差（standard deviation）で割って、どの列も「平均0・ばらつき1」の目盛りにそろえる前処理です。寝室が1つちがうことと、レビューが67件ちがうことが、同じ重さで距離に入るようになります。


### おさらい：def

```python
def 名前(引数):
    return 戻り値
```

`def`の後ろが関数の名前と引数（入力）、`return`の後ろが戻り値（出力）です。2節の`mae`も、この形で作りました。


### 変えてみる：標準化の関数を作る

- 役割：配列の各列を、決められた平均と標準偏差で標準化する
- 入力：配列`X`、列ごとの平均`mean`、列ごとの標準偏差`std`
- 出力：標準化した配列

平均と標準偏差は、訓練データだけから計算します。`axis=0`は「列ごとに」という意味です。


In [ ]:
def standardize(X, mean, std):
    return (X - mean) / std

mean_tr = X_tr.mean(axis=0)            # 列ごとの平均（訓練だけ）
std_tr = X_tr.std(axis=0)              # 列ごとの標準偏差（訓練だけ）
Z_tr = standardize(X_tr, mean_tr, std_tr)
print(Z_tr.mean(axis=0).round(2))      # どの列も平均0になったか
print(Z_tr.std(axis=0).round(2))       # どの列もばらつき1になったか


表示の`-0.`は、ごく小さなマイナスの値を小数第2位で丸めたもので、0と同じです。


### TODO③：標準化してから、似た部屋の平均で答える

テストも、訓練の平均と標準偏差で標準化し（訓練の`Z_tr`は上のセルで作りました）、お手本と同じKNNで外れ幅を表示してください。できたKNNは`knn_sc`という名前にします。


In [ ]:
# TODO: Z_te を作り、標準化した訓練で knn_sc を学び、

# 解答例③：標準化してから KNN
Z_te = standardize(X_te, mean_tr, std_tr)
knn_sc = KNeighborsRegressor(n_neighbors=10).fit(Z_tr, y_tr)
print(mae(y_te, knn_sc.predict(Z_te)), '円')


標準化すると、外れ幅は9,526円から8,315円に縮みます。目盛りをそろえただけで1,000円以上よくなり、3節の式（8,807円）も追い越しました。

scikit-learnの`StandardScaler`も、同じことをする道具です。訓練で`fit`すると、`mean_`と`scale_`に平均と標準偏差が入ります。


In [ ]:
sc = StandardScaler().fit(X_tr)
Z_te_sc = sc.transform(X_te)                     # StandardScaler で
print(np.allclose(Z_te_sc, standardize(X_te, mean_tr, std_tr)))


`True`なら、自分の関数と`StandardScaler`の答えが一致しています。線形回帰は係数で目盛りを吸収するので、標準化しても予測は変わりません。でも、距離を使うKNNでは、結果が大きく変わります。


---


## 6. 順番を守る：補う値も物差しも、訓練だけで決める


3節の中央値も、5節の平均と標準偏差も、訓練データだけから決めました。もし全データ（テストを含む）で決めると、テストの情報が前処理を通して学習に漏れます。これをデータリーク（data leakage）と呼びます。本番の問題を、こっそり過去問に混ぜるようなものです。

ここまでの前処理を1つの関数にまとめて、順番を守ったものと守らないものを比べます。


### お手本：前処理を1つの関数にまとめる

- 役割：「補う値」と「標準化の物差し」を`base`で決め、`df_in`をその値で整える
- 入力：物差しを決める表`base`、整える表`df_in`
- 出力：整えた配列

関数の中で別の関数（`standardize`）を呼べます。小さな関数を組み合わせて、大きな処理を作るのです。


In [ ]:
def prep(base, df_in):
    med = base[num].median()                 # 補う値は base で決める
    Xb = base[num].fillna(med).values
    X = df_in[num].fillna(med).values
    return standardize(X, Xb.mean(axis=0), Xb.std(axis=0))

Z_tr = prep(tr, tr)          # 正しい順番：物差しは訓練で
Z_te = prep(tr, te)
knn_ok = KNeighborsRegressor(n_neighbors=10).fit(Z_tr, y_tr)
print('訓練で決めた :', mae(y_te, knn_ok.predict(Z_te)), '円')


### 変えてみる：全データで物差しを決めると

`prep`の最初の引数を、訓練`tr`から全データ`df`に変えるだけです。


In [ ]:
Z_tr_l = prep(df, tr)        # NG：物差しを全データで決めた
Z_te_l = prep(df, te)
knn_leak = KNeighborsRegressor(n_neighbors=10).fit(Z_tr_l, y_tr)
print('全データで決めた:', mae(y_te, knn_leak.predict(Z_te_l)), '円')


全データで決めると8,322円と、正しい順番（8,315円）との差はごくわずかでした。2万件以上もあると、テストを含めても中央値や平均はほとんど動かないからです。でも、データが少ないときや、テストの期間だけ傾向が違うときには、この差が大きく出ます。結果に差が出るかどうかにかかわらず、前処理は訓練だけで決める。これを習慣にしておくことが大切です。


---


## 7. 1泊いくら？ 値付けする


6節で、補い方と物差しを関数`prep`1つにまとめました。いよいよ、自分の部屋に値段を付けます。いちばん外れ幅の小さかった、標準化したKNNで値付けします。例えば「定員4人、寝室2・ベッド2・バスルーム1」の貸切の部屋です。レビューはまだ0件、最低宿泊数は2泊、年間200日予約可能、ホストの掲載はこの1件だけとします。寝室数は分かっているので、欠けていた印は0です。新しい部屋にも、訓練で決めた同じ補い方・同じ物差しを当てはめます。関数`prep`の2つ目の引数に、新しい部屋を渡すだけです。


In [ ]:
my_room = pd.DataFrame([{'accommodates': 4, 'bedrooms': 2, 'beds': 2,
                         'bathrooms': 1, 'minimum_nights': 2,
                         'availability_365': 200,
                         'number_of_reviews': 0,
                         'host_listings_count': 1,
                         'bedrooms_missing': 0}])
price = knn_ok.predict(prep(tr, my_room))[0]
print(round(price), '円')


答えは「1泊およそ3万1千円。ただし、まだ見ていない部屋全体での外れ幅は、平均8千円ほど」。似た部屋の料金から、値付けの見当がつきました。


### TODO④：あなたの番。自分の部屋を値付けする

定員・寝室数・最低宿泊数などを自分で決めて、1泊の値付けの目安を出してください。


In [ ]:
# TODO: my_room の条件を変えた部屋を作り、prep と knn_ok で値付けする

# 解答例④：定員2人・寝室1・ベッド1の部屋
room2 = my_room.assign(accommodates=2, bedrooms=1, beds=1)
print(round(knn_ok.predict(prep(tr, room2))[0]), '円')


まだ言えないこと

- 10万円を超える部屋を外してよかったのか：外した部屋を戻すと、外れ幅はどうなるのか
- 外れ幅だけで測ってよいのか：大外れを重く見る物差しもある
- 地区の違い：4節で見たとおり、地区で料金は2倍近く違うのに、今の値付けには地区が入っていない


---


## 目標に答えられたか


- 今日の目標は「自分のマンションの一室を貸切で出すとき、1泊いくらの値段を付ければいいか」でした。7節の答えは何円で、どれくらい外れうるでしょうか？
- 1節で、10万円を超える部屋と、貸切以外の部屋を外しました。それぞれ外した理由を、今日の目標から説明できますか？
- 3節で、バスルーム数と寝室数の欠けを、それぞれどう扱いましたか？ レビューの点数の欠け方は、なぜ中央値で埋めてはいけないのでしょうか？
- 4節で、地区に番号を振って入れるのをやめたのはなぜですか？5節の標準化は、外れ幅をどれだけ縮めましたか？


---


## 今日の要点


- 外す行は、目標から理由を説明でき、どこで線を引いたかを書き残せるときに外す
- 欠けた値は、欠けた理由を先に確かめる。別の列から戻せるなら戻す
- 補うなら訓練の中央値で。補った値は推測なので、欠けていた印の列を残す
- 文字の区分に番号を振ると、振り方しだいで係数の向きまで変わる。番号の大小に意味がないから
- 距離を使うKNNでは、標準化で目盛りをそろえると、ずっとよく当たるようになる
- 補う値も標準化の物差しも、訓練データだけで決める（データリークを防ぐ）
- Python：`def`で関数を作る。引数が入力、`return`の後ろが出力。小さな関数を組み合わせて大きな処理を作る


---


## 次回へ


今日は、データを整えて、似た部屋の料金から1泊の値段を付けました。ただ、10万円を超える部屋を外したこと、外れ幅の平均だけで式を選んだことには、まだ答えていません。

次回は、場面を住宅の査定に変えます。高い家に引っぱられない当て方と、大外れを重く見る物差しを比べます。次回の問いは、「この家は、なぜこの値段なのか」を説明する式を、どの物差しで選び、どう読むか、です。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

3節では、欠けた値を訓練データの中央値で補って、線形回帰の式を立てました（`m_num`）。今度は、平均で補って（`SimpleImputer(strategy='mean')`）、同じように線形回帰の式を立ててください。そして、テストでの外れ幅（`mae`）を表示してください。


In [ ]:
imp_mean = SimpleImputer(strategy='mean').fit(tr[num])
m_mean = LinearRegression().fit(imp_mean.transform(tr[num]), y_tr)
pred = m_mean.predict(imp_mean.transform(te[num]))
print('平均で補って学習 →', mae(y_te, pred), '円')
print(pd.DataFrame({'平均': imp_mean.statistics_,
                    '中央値': imp.statistics_}, index=num).round(2))


<details><summary>詰まったら</summary>

3節の最後の`SimpleImputer(strategy='median')`を`strategy='mean'`に変え、`imp_mean`という名前にします。補う値は、3節と同じく訓練データ（`tr[num]`）で決めます。
`imp_mean.transform(tr[num])`で補った訓練の配列ができるので、それで`LinearRegression`を学び、`imp_mean.transform(te[num])`で予測します。

</details>


### 応用②（判断）

応用①で、欠けた値を平均で補ったときの、テストでの外れ幅は何円ですか。整数（円）で答えてください（例: 8123）。


In [ ]:
print('答え:', mae(y_te, pred))


<details><summary>詰まったら</summary>

2節で作った`mae(y_te, 予測)`が、四捨五入した整数を返します。3節の中央値で補ったとき（8,807円）と比べておきましょう。

</details>


### 応用③（解釈）

`imp_mean.statistics_`と`imp.statistics_`を表示すると、列ごとに、補うのに使った平均と中央値が分かります。2つを見比べて、「欠けた値を補うとき、平均と中央値のどちらを使うのがよいか」を、民泊の新しいホストに向けて3行程度で説明してください。


（模範例）

欠けた値を平均で補うと外れ幅は平均8,808円で、中央値で補ったとき（8,807円）とほとんど変わりませんでした。

ただ、補う値そのものを見ると、寝室数は平均1.55に対して中央値は1、最低宿泊数は平均4.7泊に対して中央値は1泊など、平均は極端な部屋に引っぱられています。

「寝室が1.55」のような実際にはない値より、実際にある値の中央値のほうが説明しやすいので、中央値で補うのがよいと考えます。ただし、寝室数が欠けた部屋は小さい部屋に偏っていたので、どちらで補っても推測にすぎません。欠けていた印の列を残しておき、補った部屋の値付けは幅を持って伝えるのが安全です。


---


## 発展（任意）


### 値付けの根拠になった「似た部屋」を見る

KNNの答えは、似た部屋10件の料金の平均でした。では、その10件はどんな部屋だったのでしょうか。`kneighbors`を使うと、いちばん似ていた部屋が訓練データの何行目かが分かります。7節の部屋について、似た部屋10件を並べてみます。


In [ ]:
dist, idx = knn_ok.kneighbors(prep(tr, my_room))
cols = ['neighbourhood', 'accommodates', 'bedrooms', 'beds',
        'bathrooms', 'price']
near = tr.iloc[idx[0]][cols]
print(near)
print('料金の範囲:', near['price'].min(), '〜', near['price'].max(), '円')
print('平均:', round(near['price'].mean()), '円')


読み方　似た部屋10件は、どれも定員4人前後・寝室2つ・ベッド2つで、条件はそろっています。それなのに、料金は1万5千円台から5万9千円近くまで、4倍近く開いています。並んでいる地区も、ばらばらです。

平均の約3万1千円という答えの裏には、これだけのばらつきがあるということです。ホストに答えを伝えるときは、平均だけでなく、根拠になった部屋とその幅も一緒に見せると、納得してもらいやすくなります。似た部屋の地区がばらばらなのは、今の「似ている」に地区が入っていないからです。

試すなら、`my_room`の条件を変えて、似た部屋の顔ぶれと料金の範囲がどう変わるかを見てみましょう。
